# 6. Reporte Diario — cuaderno autónomo

Cuaderno organizado para reproducir los cálculos del reporte diario y generar sus salidas desde el snapshot existente, cortes congelados y fuentes auxiliares. Incluye las funciones de rebanado, porcentajes, cortes, MJ/RIPS, indicios, gráficas y sustitución de marcadores dentro de este archivo; no importa `config.py`, `src/` ni `1_funciones/`.

**Reproducibilidad:** consume el snapshot `reporte_diario` existente, no recalcula los motores de matching. Los datos y plantilla siguen siendo insumos locales. La generación gráfica está implementada aquí, pero el rasterizado de Matplotlib puede variar por sistema, versión y fuentes; no se garantiza igualdad byte a byte de imágenes/PPTX.

## 1. Imports y rutas/configuración

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from dataclasses import dataclass, replace
from datetime import date, datetime
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from IPython.display import display, Image


def localizar_raiz(inicio: Path) -> Path:
    for ruta in (inicio.resolve(), *inicio.resolve().parents):
        if (ruta / "output" / "cache" / "universo").is_dir() and (ruta / "input" / "plantillas").is_dir():
            return ruta
    raise FileNotFoundError("No se encontró la raíz del proyecto (snapshot y plantillas).")

ROOT = localizar_raiz(Path.cwd())
SNAPSHOT_ID = "9132b089827c4e23"  # corte cacheado de reporte_diario
SNAPSHOT = ROOT / "output/cache/universo" / SNAPSHOT_ID
OUT = ROOT / "output"
RESULTADOS = OUT / "Resultados_4_Generacion_Indicadores"
GRAFICAS = ROOT / "notebooks/output"
PLANTILLA = ROOT / "input/plantillas/reporte_diario.pptx"
PPT_SALIDA = OUT / "ppt_reporte_diario" / f"Reporte de personas desaparecidas - {date.today():%d.%m.%Y}.pptx"
CORTE_MJ_RIPS = "2026-08-20"
CORTE_INDICIOS = "24jun26"
MJ_RIPS = ROOT / "input/mj/MJ y RIPS (1062).xlsx"
DIRECTORIO_INDICIOS = ROOT / "input/domicilios_y_numeros_contacto/base_combinada_con_domicilio_y_numeros_de_contacto_24062026.xlsx"
CORTES = {
 "reg_rn_25mar26_des_sisf": (ROOT/"input/datos_marzo/reg_rn_25mar26_des_sisf.xlsx", "fub", "_25mar26"),
 "reg_rn_25mar26_des_cisf_cind": (ROOT/"input/datos_marzo/reg_rn_25mar26_des_cisf_cind.xlsx", "fub", "_25mar26"),
 "reg_rn_25mar26_des_cisf_sind": (ROOT/"input/datos_marzo/reg_rn_25mar26_des_cisf_sind.xlsx", "fub", "_25mar26"),
 "reg_rn_01jul26_des_sisf": (ROOT/"input/datos_julio/reg_rn_01jul26_des_sisf.xlsx", "fub_rnpdno", "_01jul26"),
 "reg_rn_01jul26_des_cisf_cind": (ROOT/"input/datos_julio/reg_rn_01jul26_des_cisf_cind.xlsx", "fub_rnpdno", "_01jul26"),
 "reg_rn_01jul26_des_cisf_sind": (ROOT/"input/datos_julio/reg_rn_01jul26_des_cisf_sind.xlsx", "fub_rnpdno", "_01jul26"),
}
PERIODOS = {
 "min-max":(None,None),"min-01ene06":(None,"2006-01-01"),"01ene06-max":("2006-01-01",None),
 "01oct24-max":("2024-10-01",None),"01oct24-01ago25":("2024-10-01","2025-08-01"),
 "01ago25-max":("2025-08-01",None),"01ago26-max":("2026-08-01",None),"01jul26-max":("2026-07-01",None),
}
ENTIDADES={1:"Aguascalientes",2:"Baja California",3:"Baja California Sur",4:"Campeche",5:"Coahuila",6:"Colima",7:"Chiapas",8:"Chihuahua",9:"Ciudad de México",10:"Durango",11:"Guanajuato",12:"Guerrero",13:"Hidalgo",14:"Jalisco",15:"México",16:"Michoacán",17:"Morelos",18:"Nayarit",19:"Nuevo León",20:"Oaxaca",21:"Puebla",22:"Querétaro",23:"Quintana Roo",24:"San Luis Potosí",25:"Sinaloa",26:"Sonora",27:"Tabasco",28:"Tamaulipas",29:"Tlaxcala",30:"Veracruz",31:"Yucatán",32:"Zacatecas",33:"Nacido en el Extranjero"}
ESTADOS_INDICIOS=["Estado de México","Tamaulipas","Michoacán","Sinaloa","Guerrero","Veracruz","Ciudad de México","Sonora","Baja California","Zacatecas","Puebla","Nayarit","Chiapas","Tabasco","Hidalgo","San Luis Potosí","Quintana Roo","Morelos","Colima","Baja California Sur","Oaxaca","Yucatán","Tlaxcala","Campeche"]

for p in [PLANTILLA,MJ_RIPS,DIRECTORIO_INDICIOS,*[v[0] for v in CORTES.values()],SNAPSHOT/"rn.parquet",SNAPSHOT/"ba.parquet"]:
    if not p.is_file(): raise FileNotFoundError(f"Falta insumo: {p}")
RESULTADOS.mkdir(parents=True,exist_ok=True); GRAFICAS.mkdir(parents=True,exist_ok=True)
print("Raíz:",ROOT,"| Snapshot:",SNAPSHOT_ID,"| PPT:",PPT_SALIDA.name)

## 2. Funciones de rebanado, conteos, cortes y formatos

In [ ]:
BASES={"rn":("rn",None),"ba":("ba",None),"rn-no-ba":("rn",("in_banci_derivada_rnpdno_banci",False)),"rn-si-ba":("rn",("in_banci_derivada_rnpdno_banci",True)),"ba-no-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",False)),"ba-si-rn":("ba",("in_rnpdno_derivada_rnpdno_banci",True))}
DIMS={"rn":{"fecha":"fecha_hechos_modificada_derivada_rnpdno","cve_ent":"idestado_derivada_rnpdno","entidad":"nombre_entidad_derivada_rnpdno","estatus_loc":"estatus_loc_derivada_rnpdno"},"ba":{"fecha":"fha_de_hchos_derivada_banci","cve_ent":"cve_entidad_derivada_banci","entidad":"nombre_entidad_derivada_banci","estatus_loc":"estatus_loc_derivada_banci"}}
NEGACIONES={"sdel":"cdel","sind":"cind","scurp":"ccurp","cisf":"sisf","sfvac":"cfvac","sfval":"cfval"}
CONSTANTES={"rn":{},"ba":{"cind":True,"cfval":True,"cfvac":False,"fis":True,"com":False,"por":False}}
BANDERAS_ETL={"rn":{"ccurp":"curp_vic_valida_derivada_rnpdno"},"ba":{"ccurp":"curp_valida_derivada_banci"}}

@dataclass(frozen=True,order=True)
class Slice:
    base:str="rn"; periodo:str="min-max"; estatus:str="all"; subfiltro:str="all"; estado:Optional[int]=None
    def __post_init__(self):
        if self.base not in BASES or self.estatus not in {"all","des","loc"} or self.periodo not in PERIODOS: raise ValueError("Dimensiones de Slice inválidas")
        if self.estado is not None and self.estado not in ENTIDADES: raise ValueError(f"Entidad desconocida: {self.estado}")
    @property
    def nombre(self):
        p=["reg",self.base,self.periodo]
        if self.estatus!="all":p.append(self.estatus)
        if self.subfiltro!="all":p.append(self.subfiltro)
        if self.estado is not None:p.append(f"ent{self.estado:02d}")
        return "_".join(p)
    @property
    def entidad_texto(self):return ENTIDADES.get(self.estado,"Nacional") if self.estado else "Nacional"
    @property
    def padre(self):
        if self.estado is not None:return replace(self,estado=None)
        if self.subfiltro!="all":
            t=self.subfiltro.split("_")[:-1];return replace(self,subfiltro="_".join(t) if t else "all")
        if self.estatus!="all":return replace(self,estatus="all")
        return replace(self,periodo="min-max")
    def con(self,**kwargs):return replace(self,**kwargs)
    def por_entidad(self,cve):return self.con(estado=cve)

def expandir_entidades(base,entidades=None):return [base.por_entidad(c) for c in (entidades if entidades is not None else sorted(ENTIDADES))]
def col_bandera(df,token):
    cols=[c for c in df.columns if c.startswith(f"is_{token}_derivada_")]
    if len(cols)!=1:raise KeyError(f"Bandera {token}: {cols}")
    return cols[0]
def fecha_es(fecha,partes="dia_mes_anio"):
    if not fecha or not isinstance(fecha,str):return fecha
    trozos=fecha.split("-")
    if len(trozos)!=3:return fecha
    anio,mes,dia=trozos;meses={"01":"enero","02":"febrero","03":"marzo","04":"abril","05":"mayo","06":"junio","07":"julio","08":"agosto","09":"septiembre","10":"octubre","11":"noviembre","12":"diciembre"}
    valores={"dia":str(int(dia)),"mes":meses.get(mes,mes),"anio":anio}
    return " de ".join(valores[t] for t in partes.split("_"))

def normalize_header(value):
    text=unicodedata.normalize("NFD",str(value)).encode("ascii","ignore").decode("ascii").lower().strip()
    return re.sub(r"[^a-z0-9_]","",re.sub(r"[\s.\-]+","_",text))
def _fub_clean(v):return None if v is None or pd.isna(v) or str(v)=="" else str(v).strip()
def fubs_corte(marcador,verbose=True):
    ruta,col,_=CORTES[marcador]
    head=pd.read_excel(ruta,nrows=0).columns; target=normalize_header(col)
    real=next((c for c in head if normalize_header(c)==target),None)
    if real is None:raise KeyError(f"{ruta.name}: no se encontró {col}")
    values=pd.read_excel(ruta,dtype={real:str})[real]
    ids={v for x in values.dropna() if (v:=str(x).strip())}
    sin=int(values.isna().sum()+values.astype("string").str.strip().eq("").sum())
    if verbose:print(f"  📄 {marcador}: {len(ids):,} registros"+(f" (+{sin:,} sin fub)" if sin else ""))
    return ids,sin

def diferencias_corte(vars_ppt,mapa,verbose=True):
    out={}
    for actual,corte in mapa.items():
        n=int(vars_ppt[actual]);prev,_=fubs_corte(corte,verbose);key=f"{actual}-{corte}"
        out[key]=n-len(prev);out[corte]=len(prev);out[f"pct_{key}"]=f"{100*(n-len(prev))/len(prev):.1f}%" if prev else "n/d"
        if verbose:print(f"  ➖ {key} = {n:,} − {len(prev):,} = {out[key]:,}")
    return out

def deltas_corte(u,mapa,sufijo="",verbose=True):
    out={}
    for s,corte in mapa.items():
        serie=u.slice(s)["fub_rnpdno"];actuales={x for v in serie.dropna().astype(str) if (x:=v.strip())}
        prev,sin_prev=fubs_corte(corte,verbose);sin_hoy=len(serie)-len(actuales)
        altas,bajas=actuales-prev,prev-actuales
        out[f"{s.nombre}_ddes{sufijo}"]=len(altas);out[f"{s.nombre}_dloc{sufijo}"]=len(bajas)
        if sin_hoy+sin_prev:out[f"{s.nombre}_dfub{sufijo}"]=sin_hoy+sin_prev
        if verbose:print(f"  ⇄ {s.nombre} vs {corte}: +{len(altas):,} altas / −{len(bajas):,} bajas")
    return out

def porcentajes_particion(u,partes,denominador,sufijo="",decimales=1):
    ns=[u.contar(s) for s in partes];total=u.contar(denominador)
    if sum(ns)!=total:raise ValueError(f"Partes no suman {denominador.nombre}: {ns} != {total}")
    keys=[f"pct_{s.nombre}{sufijo}" for s in partes]
    if total==0:return dict.fromkeys(keys,"n/d")
    scale=100*10**decimales;qr=[divmod(scale*n,total) for n in ns];units=[q for q,r in qr];rem=[r for q,r in qr]
    for i in sorted(range(len(ns)),key=rem.__getitem__,reverse=True)[:scale-sum(units)]:units[i]+=1
    return {k:f"{n/10**decimales:.{decimales}f}%" for k,n in zip(keys,units)}

class Universo:
    """Rebanado booleano del snapshot, sin depender de src.pipeline."""
    def __init__(self,rn,ba,meta,auditoria=None):self.rn,self.ba,self.meta,self.auditoria=rn,ba,meta,auditoria;self._period_cache={}
    @classmethod
    def cargar(cls,path):
        rn=pd.read_parquet(path/"rn.parquet");ba=pd.read_parquet(path/"ba.parquet");meta=json.loads((path/"meta.json").read_text())
        aud=pd.read_csv(path/"auditoria.csv") if (path/"auditoria.csv").is_file() else None
        if len(rn)!=meta.get("n_rn") or len(ba)!=meta.get("n_ba"):raise ValueError("Snapshot incompatible con meta")
        return cls(rn,ba,meta,aud)
    @staticmethod
    def _frame(s):return BASES[s.base][0] if isinstance(s,Slice) else s
    def columna(self,s,dim):return DIMS[self._frame(s)][dim]
    def columna_bandera(self,s,token):
        frame=self._frame(s);d=self.rn if frame=="rn" else self.ba;col=BANDERAS_ETL[frame].get(token)
        return col if col else col_bandera(d,token)
    def _periodo(self,frame,p):
        key=(frame,p)
        if key in self._period_cache:return self._period_cache[key]
        d=self.rn if frame=="rn" else self.ba;fecha=d[self.columna(frame,"fecha")];ini,fin=PERIODOS[p];mask=np.ones(len(d),dtype=bool);null=fecha.isna().to_numpy()
        if ini is not None:mask &= (fecha>=pd.to_datetime(ini)).to_numpy()|null
        if fin is not None:mask &= (fecha<pd.to_datetime(fin)).to_numpy()|null
        self._period_cache[key]=mask;return mask
    def _flag(self,frame,d,t,s):
        base=NEGACIONES.get(t,t)
        vals=np.full(len(d),CONSTANTES[frame][base],dtype=bool) if base in CONSTANTES[frame] else d[self.columna_bandera(s,base)].fillna(False).astype(bool).to_numpy()
        return ~vals if t in NEGACIONES else vals
    def _mask(self,s):
        frame,member=BASES[s.base];d=self.rn if frame=="rn" else self.ba;mask=self._periodo(frame,s.periodo).copy()
        if member:
            col,val=member;b=d[col].fillna(False).to_numpy();mask &= b if val else ~b
        if s.estatus!="all":mask &= d[self.columna(s,"estatus_loc")].eq(s.estatus).fillna(False).to_numpy()
        if s.subfiltro!="all":
            for token in s.subfiltro.split("_"):mask &= self._flag(frame,d,token,s)
        if s.estado is not None:mask &= d[self.columna(s,"cve_ent")].eq(s.estado).fillna(False).to_numpy()
        return d,mask
    def contar(self,s):return int(self._mask(s)[1].sum())
    def slice(self,s):
        d,m=self._mask(s);return d[m]
    def dataframes(self,slices):return {s.nombre:self.slice(s) for s in slices}
    def rango_fechas(self,s):
        f=self._frame(s);ini,fin=PERIODOS[s.periodo];x=(self.rn if f=="rn" else self.ba)[self.columna(f,"fecha")];a,b=x.min(),x.max()
        return (a if ini is None else max(pd.Timestamp(ini),a),b if fin is None else min(pd.Timestamp(fin)-pd.Timedelta(days=1),b))
    def indicadores(self,slices,extra=None):
        out={s.nombre:self.contar(s) for s in slices};out["entidad_texto"]="Nacional"
        for base,key in (("rn","fecha_corte_rnpdno"),("ba","fecha_corte_banci")):
            fecha=self.meta.get(key)
            for p in ("dia_mes_anio","mes_anio","anio"):out[f"fha_{base}_{p}"]=fecha_es(fecha,p) if fecha else "Sin Fecha"
        hoy=datetime.now().strftime("%Y-%m-%d")
        for p in ("dia_mes_anio","dia_mes","mes_anio","anio"):out[f"fha_hoy_{p}"]=fecha_es(hoy,p)
        out.update(extra or {});return out
    def exportar_indicadores(self,slices,path,extra=None):
        df=pd.DataFrame([{"Variable":k,"Valor":v} for k,v in self.indicadores(slices,extra).items()]);path.parent.mkdir(parents=True,exist_ok=True);df.to_csv(path,index=False,encoding="utf-8-sig");print(f"📊 {len(df)} indicadores → {path}");return df
    def resumen(self):return pd.DataFrame([{"base":b,"registros":self.contar(Slice(b))} for b in BASES])

print("Helpers de cortes, particiones, fechas y Universo definidos localmente.")

## 3. Rebanadas y marcadores del reporte

In [ ]:
CONTEOS = [
    Slice("rn"),Slice("rn",estatus="des"),Slice("rn",estatus="loc"),
    *[Slice("rn",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
    Slice("rn","01oct24-max"),Slice("rn","01oct24-max",estatus="des"),Slice("rn","01oct24-max",estatus="loc"),
    *[Slice("rn","01oct24-max",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
    Slice("rn","min-01ene06",estatus="des"),Slice("rn","01ene06-max",estatus="des"),
    *[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("sisf","cisf","cisf_cind","cisf_sind")],
    *[Slice("rn","01oct24-max",estatus="des",subfiltro=t) for t in ("sisf","cisf_cind","cisf_sind")],
    *[Slice("rn-si-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("all","cpar","dfor")],
    Slice("rn-no-ba","01oct24-max",estatus="des"),
    *[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval","sfval_sfvac")],
    *[Slice("rn-no-ba",p,estatus="des",subfiltro=t) for p in ("01oct24-01ago25","01ago25-max") for t in ("cfval","sfval_sfvac")],
    *[Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro=t) for t in ("cfval","sfval_sfvac")],
    *[Slice("rn-no-ba",p,estatus="des",subfiltro=f"{f}_{i}") for p in ("01oct24-max","01ago26-max") for f in ("cfval","sfval_sfvac") for i in ("fis","com","por")],
    *[Slice("rn-no-ba",p,estatus="des",subfiltro="sfval_cfvac") for p in ("01oct24-max","01oct24-01ago25","01ago25-max")],
]
PORCENTAJES = [
    Slice("rn",estatus="des"),Slice("rn",estatus="loc"),Slice("rn","01oct24-max",estatus="des"),Slice("rn","01oct24-max",estatus="loc"),
    *[Slice("rn","01oct24-max",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
    *[Slice("rn",estatus="loc",subfiltro=t) for t in ("cdel","sdel","cvid","svid")],
    *[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("sisf","cisf")],
    *[Slice("rn-si-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cpar","dfor")],
    *[(Slice("rn","01ene06-max",estatus="des",subfiltro=t),Slice("rn","01ene06-max",estatus="des")) for t in ("cisf_cind","cisf_sind")],
    *[(Slice("rn-no-ba",p,estatus="des",subfiltro=t),Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t)) for t in ("cfval","sfval_sfvac","sfval_cfvac") for p in ("01oct24-01ago25","01ago25-max")],
]
DEN_DES=Slice("rn","01oct24-max",estatus="des");DEN_NOBA=Slice("rn-no-ba","01oct24-max",estatus="des");DEN_SFVAL=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval")
PARTICIONES=[
 ("_v1",DEN_DES,[Slice("rn-si-ba","01oct24-max",estatus="des"),DEN_NOBA]),
 ("_v1",DEN_SFVAL,[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("sfval_sfvac","sfval_cfvac")]),
 ("_v2",DEN_DES,[Slice("rn-si-ba","01oct24-max",estatus="des"),*[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval")]]),
 ("_v3",DEN_NOBA,[Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro=t) for t in ("cfval","sfval_sfvac","sfval_cfvac")]),
]
PARA_GRAFICAS=[Slice("rn-si-ba","01oct24-max",estatus="des")]
PARA_EXPORTAR=[Slice("rn","01oct24-max",estatus="des"),Slice("rn"),*[Slice("rn","01ene06-max",estatus="des",subfiltro=t) for t in ("sisf","cisf_cind","cisf_sind")]]
PARA_EXPORTAR_CON_FECHA=[Slice("rn-no-ba","01jul26-max",estatus="des",subfiltro="sfval")]
MAPA_MARZO={"reg_rn_01ene06-max_des_sisf":"reg_rn_25mar26_des_sisf","reg_rn_01ene06-max_des_cisf_cind":"reg_rn_25mar26_des_cisf_cind","reg_rn_01ene06-max_des_cisf_sind":"reg_rn_25mar26_des_cisf_sind"}
MAPA_JULIO={"reg_rn_01ene06-max_des_sisf":"reg_rn_01jul26_des_sisf","reg_rn_01ene06-max_des_cisf_cind":"reg_rn_01jul26_des_cisf_cind","reg_rn_01ene06-max_des_cisf_sind":"reg_rn_01jul26_des_cisf_sind"}
DELTAS_MARZO={Slice("rn","01ene06-max",estatus="des",subfiltro=t):c for t,c in zip(("sisf","cisf_cind","cisf_sind"),MAPA_MARZO.values())}
DELTAS_JULIO={Slice("rn","01ene06-max",estatus="des",subfiltro=t):c for t,c in zip(("sisf","cisf_cind","cisf_sind"),MAPA_JULIO.values())}
DEPENDENCIAS={"Fiscalía":"fis","Comisión":"com","Portal":"por"}
FMZ_MARK=f"reg_rn_{CORTE_INDICIOS}-max_loc_fmz"
FMZ_GRAPH=f"gr_ent_rn_{CORTE_INDICIOS}-max_loc_fmz"
ESTADOS_INDICIOS=["Estado de México","Tamaulipas","Michoacán","Sinaloa","Guerrero","Veracruz","Ciudad de México","Sonora","Baja California","Zacatecas","Puebla","Nayarit","Chiapas","Tabasco","Hidalgo","San Luis Potosí","Quintana Roo","Morelos","Colima","Baja California Sur","Oaxaca","Yucatán","Tlaxcala","Campeche"]
print(f"Definidas {len(CONTEOS)} rebanadas de conteo y {len(PORCENTAJES)} porcentajes")

## 4. Cargar universo, MJ/RIPS, indicios y cortes

In [ ]:
u=Universo.cargar(SNAPSHOT)
if u.meta.get("perfil")!="reporte_diario":raise ValueError(f"Perfil snapshot inesperado: {u.meta.get('perfil')}")
print(f"Snapshot: {u.meta.get('generado')} | RN {len(u.rn):,} | BA {len(u.ba):,} | FUBs match {u.meta.get('n_fubs_matcheados'):,}")
print(u.resumen().to_string(index=False))

# Tabla de MJ/RIPS sin deduplicar entre las dos hojas, con estado y banderas del RN.
def _normalizar_columna(x):
    t=unicodedata.normalize("NFD",str(x)).encode("ascii","ignore").decode().lower().strip()
    return re.sub(r"[^a-z0-9_]","",re.sub(r"[\s.\-]+","_",t))
def leer_excel_alias(path,aliases,sheet=0):
    raw=pd.read_excel(path,sheet_name=sheet);lookup={_normalizar_columna(c):c for c in raw.columns};rename={}
    for dest,alts in aliases.items():
        real=next((lookup[_normalizar_columna(a)] for a in alts if _normalizar_columna(a) in lookup),None)
        if real is None:raise KeyError(f"{Path(path).name}: falta {dest}; columnas {list(raw.columns)}")
        rename[real]=dest
    return raw.rename(columns=rename)[list(aliases)].copy()

MJ_MAP={"fub":["FUB","fub"],"curp":["CURP","curp"],"fecha_hechos":["fechahechos","fecha_hechos"],"nombre":["Nombre","nombre"],"apellido_paterno":["Apellido Paterno","apellido_paterno"],"apellido_materno":["Apellido Materno","apellido_materno"],"estado":["ESTADO","estado"]}
_mj=[]
for tipo,hoja in {"MJ":"MJ UNICOS","RIPS":"RIPS UNICOS"}.items():
    part=leer_excel_alias(MJ_RIPS,MJ_MAP,hoja);part["tipo"]=tipo;_mj.append(part)
mr=pd.concat(_mj,ignore_index=True,sort=False);mr["fub"]=mr["fub"].astype(str).str.strip().str.upper()
rn_est=u.rn[["fub_rnpdno","estatus_loc_derivada_rnpdno","is_cvid_derivada_rnpdno","is_svid_derivada_rnpdno"]].copy()
rn_est.columns=["fub","estatus_loc","is_cvid","is_svid"];rn_est["fub"]=rn_est["fub"].astype(str).str.strip().str.upper();rn_est=rn_est.drop_duplicates("fub")
mr=mr.merge(rn_est,on="fub",how="left");mr["estatus_loc"]=mr["estatus_loc"].fillna("des")
mr["is_cvid"]=mr["is_cvid"].fillna(False).astype(bool);mr["is_svid"]=mr["is_svid"].fillna(False).astype(bool)
mr_dfs={"reg_mr_min-max":mr,"reg_mr_min-max_des":mr[mr.estatus_loc.eq("des")],"reg_mr_min-max_loc":mr[mr.estatus_loc.eq("loc")]}
mr_dfs["reg_mr_min-max_loc_cvid"]=mr_dfs["reg_mr_min-max_loc"][mr_dfs["reg_mr_min-max_loc"].is_cvid]
mr_dfs["reg_mr_min-max_loc_svid"]=mr_dfs["reg_mr_min-max_loc"][mr_dfs["reg_mr_min-max_loc"].is_svid]
mr_vars={k:len(v) for k,v in mr_dfs.items()}
for k,den in {"reg_mr_min-max_des":"reg_mr_min-max","reg_mr_min-max_loc":"reg_mr_min-max","reg_mr_min-max_loc_cvid":"reg_mr_min-max_loc","reg_mr_min-max_loc_svid":"reg_mr_min-max_loc"}.items():
    mr_vars[f"pct_{k}"]=f"{100*mr_vars[k]/mr_vars[den]:.1f}%" if mr_vars[den] else "n/d"
for f in ("dia_mes_anio","mes_anio","anio"):mr_vars[f"fha_mr_{f}"]=fecha_es(CORTE_MJ_RIPS,f)

# Indicios de vida congelados: formalización determinada por la clasificación actual del RNPDNO.
_dir=leer_excel_alias(DIRECTORIO_INDICIOS,{"fub":["FUB","fub"],"estado_de_desaparicion":["Estado de Desaparición","estado_desaparicion","estado_de_desaparicion"]})
_dir["fub"]=_dir["fub"].astype(str).str.upper();_dir["estado_de_desaparicion"]=_dir["estado_de_desaparicion"].fillna("Sin información").replace({"México":"Estado de México"})
_clas=u.rn[["fub_rnpdno","clasificacion_des_derivada_rnpdno"]].rename(columns={"fub_rnpdno":"fub","clasificacion_des_derivada_rnpdno":"clasificacion_des"}).copy();_clas["fub"]=_clas["fub"].astype(str).str.upper()
_dir=_dir.merge(_clas,on="fub",how="left")
_es_loc=_dir["clasificacion_des"].eq("Localizados");_es_des=_dir["clasificacion_des"].isin(["Desaparecidos","Pendientes"])|_dir["clasificacion_des"].isna()
def _tab_entidad(mask,col):return _dir.loc[mask].value_counts("estado_de_desaparicion").rename_axis("estado_de_desaparicion").reset_index(name=col)
tabulado_fmz=_tab_entidad(_es_des,"todavia_desaparecidos").merge(_tab_entidad(_es_loc,"localizados"),on="estado_de_desaparicion",how="outer").fillna({"todavia_desaparecidos":0,"localizados":0})
tabulado_fmz[["todavia_desaparecidos","localizados"]]=tabulado_fmz[["todavia_desaparecidos","localizados"]].astype(int)
tabulado_fmz=tabulado_fmz.sort_values("todavia_desaparecidos",ascending=False)
renombres={"México":"Estado de México"};entidades_32=[renombres.get(n,n) for i,n in ENTIDADES.items() if i<=32]
tabulado_fmz_32=tabulado_fmz[tabulado_fmz.estado_de_desaparicion.isin(entidades_32)].reset_index(drop=True)
tabulado_fmz=tabulado_fmz[tabulado_fmz.estado_de_desaparicion.isin(ESTADOS_INDICIOS)].sort_values("todavia_desaparecidos",ascending=False).reset_index(drop=True)
fmz_nac=int(_es_loc.sum());fmz_total=int(tabulado_fmz.localizados.sum());fmz32=int(tabulado_fmz_32.localizados.sum())
fmz={"total":fmz_total,"pct":f"{100*fmz_total/fmz_nac:.1f}%" if fmz_nac else "n/d","nacional":fmz_nac,"resto":fmz_nac-fmz_total,"total_32":fmz32,"universo":int(tabulado_fmz.localizados.sum()+tabulado_fmz.todavia_desaparecidos.sum())}
print(f"MJ/RIPS {len(mr):,} filas | formalizados: {fmz_total:,} entidades participantes; país {fmz_nac:,}")

## 5. Resolver marcadores y exportar las bases de datos

In [ ]:
vars_base=u.indicadores(CONTEOS)
# Porcentajes ordinarios: usa padre salvo pares con denominador explícito.
for item in PORCENTAJES:
    s,den=item if isinstance(item,tuple) else (item,None)
    den_n=u.contar(den if den is not None else s.padre);n=u.contar(s)
    vars_base[f"pct_{s.nombre}"]=f"{100*n/den_n:.1f}%" if den_n else "n/d"
pct_particiones={k:v for suf,den,parts in PARTICIONES for k,v in porcentajes_particion(u,parts,den,suf).items()}
extra={**pct_particiones,**diferencias_corte(vars_base,MAPA_MARZO,False),**deltas_corte(u,DELTAS_MARZO,"",False),
       **diferencias_corte(vars_base,MAPA_JULIO,False),**deltas_corte(u,DELTAS_JULIO,"_v2",False),
       FMZ_MARK:fmz["total"],f"pct_{FMZ_MARK}":fmz["pct"],f"{FMZ_MARK}_v2":fmz["nacional"],
       f"{FMZ_MARK}_v3":fmz["resto"],f"{FMZ_MARK}_v4":fmz["total_32"],**mr_vars}
for periodo in ("01oct24-max","01ago26-max"):
    extra[f"fha_{periodo.split('-')[0]}_mes_anio"]=fecha_es(PERIODOS[periodo][0],"mes_anio")
vars_ppt=u.indicadores(CONTEOS,extra=extra)
print("Marcadores calculados:",len(vars_ppt))
for k,v in vars_ppt.items():print(f"  {k:<48} {v}")

dfs=u.dataframes(PARA_GRAFICAS)
exportados=u.dataframes(PARA_EXPORTAR)
for nombre,dfout in exportados.items():
    ruta=RESULTADOS/f"{nombre}.parquet";dfout.to_parquet(ruta,index=False,engine="pyarrow");print(f"💾 {ruta.name}: {len(dfout):,}")

# Entrega fechada (Parquet+XLSX), con la misma fecha DDMMYYYY en el nombre.
s_review=PARA_EXPORTAR_CON_FECHA[0];df_review=u.slice(s_review);sufijo=date.today().strftime("%d%m%Y")
nombre_fecha=f"{s_review.nombre}_{sufijo}"
df_review.to_parquet(RESULTADOS/f"{nombre_fecha}.parquet",index=False,engine="pyarrow")
df_review.to_excel(RESULTADOS/f"{nombre_fecha}.xlsx",index=False)
print(f"💾 {nombre_fecha}: {len(df_review):,} registros")

# Tablas mensuales con meses vacíos completados sobre el rango completo del período.
def agrupar_por_mes(df,col_fecha,desde=None,hasta=None):
    fechas=pd.to_datetime(df[col_fecha],errors="coerce");validas=fechas.notna()
    datos=pd.DataFrame({"mes":fechas[validas].dt.to_period("M").dt.to_timestamp()})
    conteos=datos.groupby("mes",observed=True).size().rename("registros")
    extremos=[pd.Timestamp(x).to_period("M").to_timestamp() for x in (desde,hasta) if x is not None]
    if not conteos.empty:extremos += [conteos.index.min(),conteos.index.max()]
    idx=pd.date_range(min(extremos),max(extremos),freq="MS") if extremos else pd.DatetimeIndex([])
    return pd.DataFrame({"mes":idx,"registros":conteos.reindex(idx,fill_value=0).astype(int).to_numpy()})

def serie_mensual(s):
    desde,hasta=u.rango_fechas(s);return agrupar_por_mes(u.slice(s),u.columna(s,"fecha"),desde,hasta)
def serie_mensual_apilada(series):
    return pd.concat({k:serie_mensual(s).set_index("mes")["registros"] for k,s in series.items()},axis=1).fillna(0).astype(int).rename_axis("mes").reset_index()
def tabulado_dependencia(s):
    d=u.slice(s);return pd.DataFrame({"entidad":d[u.columna(s,"entidad")],**{k:d[u.columna_bandera(s,t)] for k,t in DEPENDENCIAS.items()}}).groupby("entidad",observed=True).sum().reset_index()

def registrar(nombre,tabla):
    total=tabla.select_dtypes("number").to_numpy().sum();print(f"  📊 {nombre}: {total:,} registros / {len(tabla)} filas")
    GRAFICAS_GENERADAS.append(nombre)
    if tabla.empty or tabla.select_dtypes("number").to_numpy().sum()==0:GRAFICAS_SIN_DATOS.append(nombre)
    display(Image(filename=str(GRAFICAS/f"{nombre}.png")))


## 6. Gráficas y ensamblado PPTX (funciones locales)

In [ ]:
GRAFICAS_GENERADAS=[];GRAFICAS_SIN_DATOS=[]
def dimensiones_placeholder(pptx_path,clave):
    objetivo=f"{{{{{clave}}}}}";fact=914400/2.54;found=[]
    def walk(shapes):
        for sh in shapes:
            if sh.shape_type==MSO_SHAPE_TYPE.GROUP:walk(sh.shapes)
            elif sh.has_text_frame and sh.text.strip()==objetivo:found.append(sh)
    for slide in Presentation(str(pptx_path)).slides:walk(slide.shapes)
    if not found:raise KeyError(f"No se encuentra placeholder {objetivo}")
    return found[0].width/fact,found[0].height/fact

def _guardar_fig(fig,nombre):
    fig.savefig(GRAFICAS/f"{nombre}.png",dpi=300,transparent=True)
    fig.savefig(GRAFICAS/f"{nombre}.svg",transparent=True)
    plt.close(fig)

def _aviso(fig,ax,nombre):
    if not ax.has_data():
        GRAFICAS_SIN_DATOS.append(nombre);ax.clear();ax.text(.5,.5,"Sin registros en el período",ha="center",va="center",transform=ax.transAxes,color="#757575",fontsize=14);ax.axis("off")

def grafica_mensual(s,nombre,color,apiladas=None):
    if apiladas is None:
        tabla=serie_mensual(s);series={"registros":tabla.registros}
    else:
        tabla=serie_mensual_apilada(apiladas);series={c:tabla[c] for c in tabla.columns[1:]}
    w,h=dimensiones_placeholder(PLANTILLA,nombre);fig,ax=plt.subplots(figsize=(w/2.54,h/2.54),dpi=300)
    colors=color if isinstance(color,list) else [color]
    x=np.arange(len(tabla));bottom=np.zeros(len(tabla))
    for i,(label,values) in enumerate(series.items()):
        vals=np.asarray(values,dtype=float);bars=ax.bar(x,vals,bottom=bottom,color=colors[i%len(colors)],width=.85,label=label)
        if apiladas is not None:
            for j,v in enumerate(vals):
                if v>0:ax.text(j,bottom[j]+v/2,f"{int(v):,}",ha="center",va="center",fontsize=6,color="white")
        bottom+=vals
    if len(tabla):
        step=max(1,len(tabla)//9);ax.set_xticks(x[::step]);ax.set_xticklabels(pd.to_datetime(tabla.mes).dt.strftime("%b-%y").iloc[::step],fontsize=7)
        if apiladas is None:
            for j,v in enumerate(bottom):
                if v:ax.text(j,v,f"{int(v):,}",ha="center",va="bottom",fontsize=6)
        else:
            for j,v in enumerate(bottom):
                if v:ax.text(j,v,f"{int(v):,}",ha="center",va="bottom",fontsize=6)
    ax.grid(axis="y",alpha=.2);ax.set_axisbelow(True);ax.spines[["top","right"]].set_visible(False);fig.tight_layout()
    _aviso(fig,ax,nombre);_guardar_fig(fig,nombre);registrar(nombre,tabla)

def grafica_horizontal(tabla,nombre,cols,colores,ancho=20,alto=12,valor_barra=True):
    fig,ax=plt.subplots(figsize=(ancho/2.54,alto/2.54),dpi=300)
    if not tabla.empty:
        yy=np.arange(len(tabla));left=np.zeros(len(tabla))
        for col,color in zip(cols,colores):
            vals=pd.to_numeric(tabla[col],errors="coerce").fillna(0).to_numpy();ax.barh(yy,vals,left=left,color=color,label=col)
            if valor_barra:
                for j,v in enumerate(vals):
                    if v and v>max(1,left.max()*.02):ax.text(left[j]+v/2,j,f"{int(v):,}",ha="center",va="center",fontsize=7,color="white")
            left+=vals
        ax.set_yticks(yy);ax.set_yticklabels(tabla.iloc[:,0].astype(str),fontsize=7);ax.invert_yaxis()
        for j,v in enumerate(left):
            if v:ax.text(v,j,f"{int(v):,}",ha="left",va="center",fontsize=7)
    ax.grid(axis="x",alpha=.2);ax.set_axisbelow(True);ax.spines[["top","right","left"]].set_visible(False);fig.tight_layout()
    _aviso(fig,ax,nombre);_guardar_fig(fig,nombre);registrar(nombre,tabla)

def registrar(nombre,tabla):
    print(f"  📊 {nombre}: {int(tabla.select_dtypes('number').to_numpy().sum()):,} registros en {len(tabla)} filas")
    GRAFICAS_GENERADAS.append(nombre)
    if tabla.empty or tabla.select_dtypes("number").to_numpy().sum()==0:GRAFICAS_SIN_DATOS.append(nombre)
    display(Image(filename=str(GRAFICAS/f"{nombre}.png")))

def _serie_dependencia(s):
    d=u.slice(s)
    return (pd.DataFrame({"entidad":d[u.columna(s,"entidad")],**{k:d[u.columna_bandera(s,t)] for k,t in DEPENDENCIAS.items()}})
            .groupby("entidad",observed=True).sum().reset_index())

# Seis gráficas mensuales requeridas.
S_BA=Slice("rn-si-ba","01oct24-max",estatus="des")
S_CF=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="cfval")
S_SF=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval")
S_SFV=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval_sfvac")
S_SFC=Slice("rn-no-ba","01oct24-max",estatus="des",subfiltro="sfval_cfvac")
grafica_mensual(S_BA,"gr_mes_rn-si-ba_01oct24-max_des","#af5967ff")
grafica_mensual(S_CF,"gr_mes_rn-no-ba_01oct24-max_des_cfval","#9b2347ff")
grafica_mensual(S_SF,"gr_mes_rn-no-ba_01oct24-max_des_sfval","#595959ff")
grafica_mensual(S_SFV,"gr_mes_rn-no-ba_01oct24-max_des_sfval_sfvac","#757575ff")
grafica_mensual(S_SFC,"gr_mes_rn-no-ba_01oct24-max_des_sfval_cfvac","#888888ff")
grafica_mensual(S_BA,"gr_mes_rn_01oct24-max_des_sisf-cind-sind",["#ed7d31ff","#2a7473ff","#56408fff"],{
 "Datos insuficientes":Slice("rn","01oct24-max",estatus="des",subfiltro="sisf"),
 "Con actividades":Slice("rn","01oct24-max",estatus="des",subfiltro="cisf_cind"),
 "Sin actividad":Slice("rn","01oct24-max",estatus="des",subfiltro="cisf_sind")})
for s,n in [(S_CF,"gr_ent_rn-no-ba_01oct24-max_des_cfval"),(S_SFV,"gr_ent_rn-no-ba_01oct24-max_des_sfval_sfvac"),
            (Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro="cfval"),"gr_ent_rn-no-ba_01ago26-max_des_cfval"),
            (Slice("rn-no-ba","01ago26-max",estatus="des",subfiltro="sfval_sfvac"),"gr_ent_rn-no-ba_01ago26-max_des_sfval_sfvac")]:
    grafica_horizontal(_serie_dependencia(s),n,["Fiscalía","Comisión","Portal"],["#4f3130","#e44d03","#de639b"])
grafica_horizontal(tabulado_fmz,FMZ_GRAPH,["todavia_desaparecidos","localizados"],["#32485f","#b3c5bf"])
grafica_horizontal(tabulado_fmz_32,f"gr_32ent_rn_{CORTE_INDICIOS}-max_loc_fmz",["todavia_desaparecidos","localizados"],["#32485f","#b3c5bf"])
print(f"Total de gráficas generadas: {len(GRAFICAS_GENERADAS)}")

### Sustitución de variables de texto e imágenes en plantilla

In [ ]:
class PPTReportBuilder:
    """Sustituye marcadores de texto y cuadros de imagen de la plantilla PPTX."""
    def __init__(self,template_path):
        self.template_path=Path(template_path)
        if not self.template_path.exists():raise FileNotFoundError(f"La plantilla PPTX no existe: {self.template_path}")
        self.prs=Presentation(str(self.template_path));self.text_replacements={};self.image_replacements={}
    @staticmethod
    def format_value(key,value):
        if value is None:return ""
        if any(x in str(key).lower() for x in ("año","anio","year")):
            try:return str(int(float(value)))
            except (ValueError,TypeError):return str(value)
        if isinstance(value,str):return value
        try:
            v=float(value);return f"{v:,.0f}" if v.is_integer() else str(value)
        except (ValueError,TypeError):return str(value)
    def add_text_variables(self,variables):
        self.text_replacements={str(k).strip():self.format_value(k,v) for k,v in variables.items()};return self
    def add_image_variables(self,images):self.image_replacements={str(k).strip():str(v) for k,v in images.items()};return self
    def _replace_paragraph(self,p):
        counts={k:0 for k in self.text_replacements};text=p.text
        for k,v in self.text_replacements.items():
            marker=f"{{{{{k}}}}}"
            if marker in text:
                counts[k]=text.count(marker);text=text.replace(marker,v)
        if any(counts.values()) and p.runs:
            p.runs[0].text=text
            for r in p.runs[1:]:r.text=""
        return counts
    def _shape_text(self,shape):
        counts={k:0 for k in self.text_replacements}
        if shape.has_text_frame:
            for p in shape.text_frame.paragraphs:
                for k,v in self._replace_paragraph(p).items():counts[k]+=v
        elif shape.has_table:
            for row in shape.table.rows:
                for cell in row.cells:
                    for p in cell.text_frame.paragraphs:
                        for k,v in self._replace_paragraph(p).items():counts[k]+=v
        elif shape.shape_type==MSO_SHAPE_TYPE.GROUP:
            for sub in shape.shapes:
                for k,v in self._shape_text(sub).items():counts[k]+=v
        return counts
    def render(self,output_path,verbose=True):
        text_report={k:0 for k in self.text_replacements};image_report={k:0 for k in self.image_replacements}
        for slide in self.prs.slides:
            for shape in slide.shapes:
                for k,v in self._shape_text(shape).items():text_report[k]+=v
            for shape in list(slide.shapes):
                if not shape.has_text_frame:continue
                for key,path in self.image_replacements.items():
                    if shape.text.strip()!=f"{{{{{key}}}}}":continue
                    if not Path(path).exists():print(f"⚠️ Imagen ausente: {path}");break
                    x,y,w,h=shape.left,shape.top,shape.width,shape.height
                    element=shape.element;element.getparent().remove(element)
                    slide.shapes.add_picture(str(path),x,y,w,h);image_report[key]+=1;break
        output_path=Path(output_path);output_path.parent.mkdir(parents=True,exist_ok=True);self.prs.save(str(output_path))
        if verbose:
            print(f"✅ Presentación guardada: {output_path}")
            print(f"  📝 Textos reemplazados: {sum(v>0 for v in text_report.values())}/{len(text_report)}")
            print(f"  🖼️ Imágenes reemplazadas: {sum(v>0 for v in image_report.values())}/{len(image_report)}")
        return {"output_path":output_path,"text_report":text_report,"image_report":image_report}

builder=PPTReportBuilder(PLANTILLA).add_text_variables(vars_ppt).add_image_variables({n:GRAFICAS/f"{n}.png" for n in GRAFICAS_GENERADAS})
reporte=builder.render(PPT_SALIDA)
faltantes_texto=[k for k,v in reporte["text_report"].items() if v==0]
faltantes_imagen=[k for k,v in reporte["image_report"].items() if v==0]
if faltantes_texto:print("Marcadores de texto no encontrados:",faltantes_texto)
if faltantes_imagen:print("Placeholders de imagen no encontrados:",faltantes_imagen)
if GRAFICAS_SIN_DATOS:print("Gráficas vacías, muestran aviso:",GRAFICAS_SIN_DATOS)
assert PPT_SALIDA.is_file() and PPT_SALIDA.stat().st_size>0